# 샘플링 CSV -> 라벨링 입력 포맷(jsonl) 변환

윤서가 만든 `sampled_dataset.csv`(occupation, gender, experience, file_path, question, answer)를
`gemini_batch_labeling.ipynb`가 기대하는 jsonl 포맷({id, job, question, answer})으로 변환.

occupation 코드(`01.Management` 등)는 한글 직무명으로 매핑해서 넣음.

## 1. Google Drive 마운트
csv 파일이 Drive에 있으면 실행

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. 설정값

In [ ]:
INPUT_CSV = "/content/sampled_dataset.csv"      # 실제 경로로 수정 (Drive 업로드했다면 /content/drive/... 경로)
OUTPUT_PATH = "/content/sampled_dataset.jsonl"      # 라벨링 노트북 INPUT_PATH에 넣을 결과 파일

## 3. 임포트 & 직군 매핑

In [ ]:
import csv
import json
import re

# 실제 다운로드 폴더명 기준 7개 직군 매핑
CATEGORY_MAP = {
    "Management": "경영/사무",
    "SalesMarketing": "영업/마케팅",
    "PublicService": "공공/서비스",
    "RND": "연구개발",
    "ICT": "ICT",
    "Design": "디자인",
    "ProductionManufacturing": "생산/제조",
}

CATEGORY_PATTERN = re.compile(r"\d{2}\.([A-Za-z]+)")


def resolve_job_name(occupation_code: str) -> str:
    match = CATEGORY_PATTERN.search(occupation_code or "")
    category = match.group(1) if match else occupation_code
    return CATEGORY_MAP.get(category, category or "미상")

## 4. 변환 실행

In [ ]:
with open(INPUT_CSV, "r", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print(f"입력 CSV 총 {len(rows)}건")

converted = 0
skipped = 0
with open(OUTPUT_PATH, "w", encoding="utf-8") as out_f:
    for i, row in enumerate(rows, 1):
        question = (row.get("question") or "").strip()
        answer = (row.get("answer") or "").strip()
        if not question or not answer:
            skipped += 1
            continue

        record = {
            "id": i,
            "job": resolve_job_name(row.get("occupation", "")),
            "question": question,
            "answer": answer,
            "meta": {
                "occupation_code": row.get("occupation", ""),
                "gender": row.get("gender", ""),
                "experience": row.get("experience", ""),
                "file_path": row.get("file_path", ""),
            },
        }
        out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
        converted += 1

print(f"변환 완료: {converted}건 저장, {skipped}건 스킵 (질문/답변 비어있음) -> {OUTPUT_PATH}")

입력 CSV 총 1999건
변환 완료: 1999건 저장, 0건 스킵 (질문/답변 비어있음) -> /content/sampled_dataset.jsonl


## 5. (선택) 결과 미리 확인 + 직군 분포 체크

In [ ]:
from collections import Counter

with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    converted_rows = [json.loads(line) for line in f]

print("직군 분포:", Counter(r["job"] for r in converted_rows))
print()
for r in converted_rows[:2]:
    print(r["id"], "|", r["job"], "|", r["question"][:50])

직군 분포: Counter({'영업/마케팅': 598, '경영/사무': 578, 'ICT': 307, '공공/서비스': 286, '연구개발': 121, '디자인': 98, '생산/제조': 11})

1 | 경영/사무 | 귀하께서는 전 직장에 약 한 삼 년 동안 근무를 하셨는데 전 직장에서 기억에 남는 프로젝트
2 | 경영/사무 | 직전 회사를 그만두신 이유가 있으실 겁니다 혹시 동일한 이유로 현 직장도 그만둘 가능성이 
